In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fichka/almaty-air-quality-history")

print("Path to dataset files:", path)

Path to dataset files: /Users/zadyra/.cache/kagglehub/datasets/fichka/almaty-air-quality-history/versions/2


In [4]:
import os
import pandas as pd

In [5]:
print(os.listdir(path))

['locations_else.csv', 'locations_air_gradient.csv', 'air_quality_data.csv']


# Almaty Air Quality Dataset

| Column | Description |
|---|---|
| `datetime` | Hourly timestamp (UTC) |
| `location_id` | Unique sensor station ID |
| `pm25` | PM2.5 particles <2.5µm — main health metric (µg/m³) |
| `pm1` | Ultra-fine particles <1µm (µg/m³) |
| `pm10` | Coarser particles <10µm (µg/m³) |
| `relativehumidity` | Humidity % |
| `temperature` | Temperature °C |
| `um003` | Particle count >0.3µm per 0.1L of air |
| `provider_name` | AirNow (US Embassy), Clarity (sensor network), AirGradient (newer sensors) |

**WHO thresholds for PM2.5:** Good <12 · Moderate 12–35 · Unhealthy 35–55 · Very Unhealthy 55–150 · Hazardous >150
**Date range:** Apr 2020 – Jan 2026  |  **Sensors:** 140+ locations  |  **Rows:** 544,943

## Load & Clean

In [7]:
import pandas as pd, os, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(os.path.join(path, 'air_quality_data.csv'))
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

# Time features
df['year']    = df['datetime'].dt.year
df['month']   = df['datetime'].dt.month
df['hour']    = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.dayofweek   # 0=Mon, 6=Sun
df['season']  = df['month'].map({12:'Winter',1:'Winter',2:'Winter',
                                  3:'Spring',4:'Spring',5:'Spring',
                                  6:'Summer',7:'Summer',8:'Summer',
                                  9:'Autumn',10:'Autumn',11:'Autumn'})

# AQI category based on PM2.5
def aqi_cat(v):
    if pd.isna(v):   return 'Unknown'
    if v < 12:       return '1-Good'
    if v < 35.5:     return '2-Moderate'
    if v < 55.5:     return '3-Unhealthy for Sensitive'
    if v < 150.5:    return '4-Unhealthy'
    return '5-Hazardous'
df['aqi_cat'] = df['pm25'].apply(aqi_cat)

print(f'Shape: {df.shape}')
print(f'Date range: {df["datetime"].min().date()} → {df["datetime"].max().date()}')
print(f'\nProvider breakdown:')
print(df['provider_name'].value_counts().to_string())
print(f'\nNull counts:')
print(df[['pm25','pm1','pm10','relativehumidity','temperature']].isnull().sum().to_string())


Shape: (544943, 18)
Date range: 2020-04-09 → 2026-01-07

Provider breakdown:
provider_name
Clarity        446970
AirNow          49076
AirGradient     48897

Null counts:
pm25                 13990
pm1                 496046
pm10                534740
relativehumidity    496046
temperature         496046


In [8]:
df

,datetime,location_id,pm25,pm1,pm10,relativehumidity,temperature,um003,name,lat,lon,provider_name,year,month,hour,weekday,season,aqi_cat
0,2020-04-09 16:00:00+00:00,8876,6.000000,NaN,NaN,NaN,NaN,NaN,Almaty,43.252855,76.93118,AirNow,2020,4,16,3,Spring,1-Good
1,2020-04-09 17:00:00+00:00,8876,10.000000,NaN,NaN,NaN,NaN,NaN,Almaty,43.252855,76.93118,AirNow,2020,4,17,3,Spring,1-Good
2,2020-04-09 18:00:00+00:00,8876,8.000000,NaN,NaN,NaN,NaN,NaN,Almaty,43.252855,76.93118,AirNow,2020,4,18,3,Spring,1-Good
3,2020-04-09 19:00:00+00:00,8876,6.000000,NaN,NaN,NaN,NaN,NaN,Almaty,43.252855,76.93118,AirNow,2020,4,19,3,Spring,1-Good
4,2020-04-09 20:00:00+00:00,8876,4.000000,NaN,NaN,NaN,NaN,NaN,Almaty,43.252855,76.93118,AirNow,2020,4,20,3,Spring,1-Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544938,2026-01-07 10:00:00+00:00,6186958,93.809091,54.886364,NaN,56.422727,3.772727,3554.227273,"Almaty, Dostyk 300/60",43.222600,76.97590,AirGradient,2026,1,10,2,Winter,4-Unhealthy
544939,2026-01-07 11:00:00+00:00,6186958,98.137500,56.687500,NaN,63.466667,1.866667,3563.833333,"Almaty, Dostyk 300/60",43.222600,76.97590,AirGradient,2026,1,11,2,Winter,4-Unhealthy
544940,2026-01-07 12:00:00+00:00,6186958,86.840000,50.335000,NaN,67.030000,-0.040000,3068.650000,"Almaty, Dostyk 300/60",43.222600,76.97590,AirGradient,2026,1,12,2,Winter,4-Unhealthy
544941,2026-01-07 13:00:00+00:00,6186958,42.850000,24.036363,NaN,56.945454,-0.686364,1493.772727,"Almaty, Dostyk 300/60",43.222600,76.97590,AirGradient,2026,1,13,2,Winter,3-Unhealthy for Sensitive


---
## 1 — Annual PM2.5 Trends (Is air quality improving?)

In [9]:
annual = df[df['pm25'].notna()].groupby('year').agg(
    avg_pm25   =('pm25','mean'),
    median_pm25=('pm25','median'),
    max_pm25   =('pm25','max'),
    readings   =('pm25','count'),
).round(2)

# % of readings in each WHO category per year
who_15  = df[df['pm25'].notna()].groupby('year').apply(lambda x: (x['pm25']>15).mean()*100).round(1)
who_55  = df[df['pm25'].notna()].groupby('year').apply(lambda x: (x['pm25']>55).mean()*100).round(1)
who_150 = df[df['pm25'].notna()].groupby('year').apply(lambda x: (x['pm25']>150).mean()*100).round(1)
annual['pct_above_15']  = who_15
annual['pct_above_55']  = who_55
annual['pct_above_150'] = who_150

print('Annual PM2.5 summary (µg/m³):')
print(annual.to_string())
best = annual['avg_pm25'].idxmin()
worst = annual['avg_pm25'].idxmax()
print(f'\nBest year:  {best} (avg {annual.loc[best,"avg_pm25"]:.1f} µg/m³)')
print(f'Worst year: {worst} (avg {annual.loc[worst,"avg_pm25"]:.1f} µg/m³)')


Annual PM2.5 summary (µg/m³):
      avg_pm25  median_pm25  max_pm25  readings  pct_above_15  pct_above_55  pct_above_150
year                                                                                      
2020     29.20        13.00    337.00      6392          43.5          14.3            3.0
2021     24.57        10.00    396.00      8760          34.5          11.8            1.9
2022     40.54        32.97    375.00      8760          83.2          18.8            1.2
2023     31.45        14.00    342.00      8760          46.7          18.4            2.5
2024     35.44        13.65    999.40    216421          45.0           9.4            3.4
2025     48.98        16.42    998.66    264854          55.0          16.6            5.1
2026     56.83        53.06    479.64     17006          85.2          46.0            3.5

Best year:  2021 (avg 24.6 µg/m³)
Worst year: 2026 (avg 56.8 µg/m³)


---
## 2 — Seasonal & Monthly Patterns

In [10]:
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

monthly = df[df['pm25'].notna()].groupby('month')['pm25'].agg(
    avg='mean', median='median', max='max'
).round(2)
monthly.index = monthly.index.map(month_names)

print('Monthly PM2.5 averages:')
print(monthly.to_string())

print('\nSeasonal averages:')
seasonal = df[df['pm25'].notna()].groupby('season')['pm25'].agg(
    avg='mean', median='median', pct_unhealthy=lambda x: (x>55).mean()*100
).round(2)
print(seasonal.sort_values('avg', ascending=False).to_string())

print(f'\nWorst month: {monthly["avg"].idxmax()} ({monthly["avg"].max():.1f} µg/m³)')
print(f'Best month:  {monthly["avg"].idxmin()} ({monthly["avg"].min():.1f} µg/m³)')


Monthly PM2.5 averages:
         avg  median     max
month                       
Jan    55.91   40.80  959.15
Feb    60.16   47.07  971.29
Mar    29.69   22.42  982.33
Apr    26.93   13.42  996.82
May    30.36   10.89  999.40
Jun    28.78   12.76  991.58
Jul    40.10   12.29  998.66
Aug    41.34   12.29  991.24
Sep    39.01   13.14  998.34
Oct    45.54   16.92  992.30
Nov    53.38   25.12  999.27
Dec    64.87   47.69  990.05

Seasonal averages:
          avg  median  pct_unhealthy
season                              
Winter  60.52   44.91          41.47
Autumn  45.02   16.36           9.66
Summer  36.77   12.47           4.52
Spring  29.22   13.15           4.96

Worst month: Dec (64.9 µg/m³)
Best month:  Apr (26.9 µg/m³)


---
## 3 — Hourly & Weekday Patterns (Rush hours & heating)

In [11]:
print('PM2.5 by hour of day:')
hourly = df[df['pm25'].notna()].groupby('hour')['pm25'].mean().round(2)
print(hourly.to_string())
print(f'  Peak hour:   {hourly.idxmax()}:00 ({hourly.max():.2f} µg/m³)')
print(f'  Cleanest:    {hourly.idxmin()}:00 ({hourly.min():.2f} µg/m³)')

print('\nPM2.5 by day of week:')
day_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
daily = df[df['pm25'].notna()].groupby('weekday')['pm25'].mean().round(2)
daily.index = daily.index.map(day_names)
print(daily.to_string())

print('\nWeekend vs Weekday:')
df['is_weekend'] = df['weekday'] >= 5
wknd = df[df['pm25'].notna()].groupby('is_weekend')['pm25'].agg(
    avg='mean', median='median'
).round(2)
wknd.index = wknd.index.map({False:'Weekday',True:'Weekend'})
print(wknd.to_string())


PM2.5 by hour of day:
hour
0     37.39
1     36.96
2     37.05
3     37.22
4     37.45
5     38.10
6     38.42
7     39.06
8     40.14
9     40.89
10    41.56
11    42.93
12    45.31
13    48.49
14    50.96
15    51.80
16    51.42
17    49.90
18    48.20
19    46.03
20    43.84
21    41.92
22    39.97
23    38.39
  Peak hour:   15:00 (51.80 µg/m³)
  Cleanest:    1:00 (36.96 µg/m³)

PM2.5 by day of week:
weekday
Mon    43.90
Tue    41.88
Wed    42.79
Thu    40.14
Fri    39.81
Sat    44.05
Sun    45.97

Weekend vs Weekday:
              avg  median
is_weekend               
Weekday     41.71   15.08
Weekend     45.01   16.92


---
## 4 — WHO Threshold Exceedance Days
WHO 24-hour PM2.5 limit = 15 µg/m³

In [12]:
# Daily averages from the main AirNow sensor (longest continuous record)
airnow = df[df['provider_name']=='AirNow'].copy()
airnow['date'] = airnow['datetime'].dt.date
daily_avg = airnow.groupby(['year','date'])['pm25'].mean().reset_index()
daily_avg.columns = ['year','date','daily_pm25']

print('Days exceeding WHO 24h limit (15 µg/m³) per year — AirNow sensor:')
exc = daily_avg.groupby('year').apply(lambda x: pd.Series({
    'total_days'    : len(x),
    'days_above_15' : (x['daily_pm25']>15).sum(),
    'days_above_35' : (x['daily_pm25']>35).sum(),
    'days_above_55' : (x['daily_pm25']>55).sum(),
    'pct_above_15'  : round((x['daily_pm25']>15).mean()*100, 1),
})).astype({'total_days':int,'days_above_15':int,'days_above_35':int,'days_above_55':int})
print(exc.to_string())

print('\nAQI category distribution (all providers, all years):')
print(df[df['aqi_cat']!='Unknown']['aqi_cat'].value_counts().sort_index().to_string())
print(f'\n% of readings that are Good (<12 µg/m³): {(df["pm25"]<12).sum()/(df["pm25"].notna()).sum()*100:.1f}%')


Days exceeding WHO 24h limit (15 µg/m³) per year — AirNow sensor:
      total_days  days_above_15  days_above_35  days_above_55  pct_above_15
year                                                                       
2020         267            120             55             39          44.9
2021         365            131             69             46          35.9
2022         365            330             93             70          90.4
2023         365            191             90             67          52.3
2024         366            170             86             46          46.4
2025         318            168             58             36          52.8

AQI category distribution (all providers, all years):
aqi_cat
1-Good                       187070
2-Moderate                   214224
3-Unhealthy for Sensitive     53062
4-Unhealthy                   54431
5-Hazardous                   22166

% of readings that are Good (<12 µg/m³): 35.2%


---
## 5 — Temperature & Humidity vs PM2.5
(Cold winters = coal/heating = worse air?)

In [13]:
ag = df[df['provider_name']=='AirGradient'].dropna(subset=['pm25','temperature','relativehumidity'])

print('Correlation matrix (AirGradient sensors, which have all fields):')
print(ag[['pm25','pm1','temperature','relativehumidity','um003']].corr().round(3).to_string())

print('\nAvg PM2.5 by temperature bracket:')
ag2 = ag.copy()
ag2['temp_bin'] = pd.cut(ag2['temperature'], bins=[-10,0,5,10,15,20,30,45],
                         labels=['<0°C','0-5°C','5-10°C','10-15°C','15-20°C','20-30°C','>30°C'])
print(ag2.groupby('temp_bin')['pm25'].agg(avg='mean',median='median',count='count').round(2).to_string())

print('\nAvg PM2.5 by humidity bracket:')
ag2['hum_bin'] = pd.cut(ag2['relativehumidity'], bins=[0,30,40,50,60,70,100],
                        labels=['<30%','30-40%','40-50%','50-60%','60-70%','>70%'])
print(ag2.groupby('hum_bin')['pm25'].agg(avg='mean',median='median',count='count').round(2).to_string())


Correlation matrix (AirGradient sensors, which have all fields):
                   pm25    pm1  temperature  relativehumidity  um003
pm25              1.000  0.995       -0.267             0.219  0.697
pm1               0.995  1.000       -0.251             0.202  0.703
temperature      -0.267 -0.251        1.000            -0.678 -0.070
relativehumidity  0.219  0.202       -0.678             1.000  0.105
um003             0.697  0.703       -0.070             0.105  1.000

Avg PM2.5 by temperature bracket:
            avg  median  count
temp_bin                      
<0°C      53.22   47.13    183
0-5°C     67.73   61.53   4166
5-10°C    69.36   59.27  29027
10-15°C   64.47   50.63   8704
15-20°C   43.77   32.11   2880
20-30°C   17.79   14.18   3471
>30°C     14.99   14.49    466

Avg PM2.5 by humidity bracket:
           avg  median  count
hum_bin                      
<30%     20.87   14.22   2987
30-40%   40.69   27.45   8316
40-50%   77.40   66.41  17965
50-60%   65.73   55.57  1

---
## 6 — Worst Pollution Episodes

In [14]:
print('Top 15 most polluted hourly readings (PM2.5 µg/m³):')
top = df[df['pm25']<999].nlargest(15,'pm25')[['datetime','location_id','pm25','provider_name','season','year']]
print(top.to_string(index=False))

print('\nTop 10 worst daily averages (AirNow):')
print(daily_avg.nlargest(10,'daily_pm25').to_string(index=False))

print('\nHazardous days (daily avg >150 µg/m³) by month across all years:')
haz = daily_avg[daily_avg['daily_pm25']>150].copy()
haz['month'] = pd.to_datetime(haz['date']).dt.month
print(haz['month'].map({1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                         7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}).value_counts().to_string())


Top 15 most polluted hourly readings (PM2.5 µg/m³):
                 datetime  location_id       pm25 provider_name season  year
2024-05-18 13:00:00+00:00      2812716 998.958421       Clarity Spring  2024
2024-05-18 14:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 15:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 16:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 17:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 18:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 19:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 20:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 21:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 22:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-18 23:00:00+00:00      2812716 998.847193       Clarity Spring  2024
2024-05-19 00:00:00+00:0

---
## 7 — Sensor Location Comparison (Which areas are worst?)

In [15]:
clarity = df[df['provider_name']=='Clarity'].copy()

loc_stats = clarity.groupby('location_id').agg(
    avg_pm25     =('pm25','mean'),
    median_pm25  =('pm25','median'),
    pct_unhealthy=('pm25', lambda x: (x>55).mean()*100),
    lat          =('lat','mean'),
    lon          =('lon','mean'),
    readings     =('pm25','count'),
).round(3).sort_values('avg_pm25', ascending=False)

print('Clarity sensor locations — ranked by avg PM2.5:')
print(loc_stats[['avg_pm25','median_pm25','pct_unhealthy','lat','lon','readings']].to_string())

print(f'\nDirtiest sensor: location_id {loc_stats.index[0]} '
      f'(lat={loc_stats.iloc[0]["lat"]:.4f}, lon={loc_stats.iloc[0]["lon"]:.4f}, '
      f'avg PM2.5={loc_stats.iloc[0]["avg_pm25"]:.2f} µg/m³)')
print(f'Cleanest sensor: location_id {loc_stats.index[-1]} '
      f'(lat={loc_stats.iloc[-1]["lat"]:.4f}, lon={loc_stats.iloc[-1]["lon"]:.4f}, '
      f'avg PM2.5={loc_stats.iloc[-1]["avg_pm25"]:.2f} µg/m³)')


Clarity sensor locations — ranked by avg PM2.5:
             avg_pm25  median_pm25  pct_unhealthy     lat     lon  readings
location_id                                                                
2812691       232.361       39.525         44.742  43.297  76.861     12952
2812649       232.243       26.932         38.492  43.283  76.928     14060
2812753       126.274       11.374         21.178  43.155  76.899     12952
2812716        86.979       14.052         20.259  43.228  76.960     12952
2812821        42.470       18.700         23.728  43.336  76.913      8016
3124030        40.893       41.321          3.076  43.308  76.923      7021
2819078        36.954       36.669          1.269  43.302  76.872     12765
2812651        34.986       19.290         16.901  43.291  76.841     12952
2812689        31.873       15.591         14.739  43.342  76.982     12952
2819013        31.447       23.998          6.648  43.229  76.777     12650
2812563        30.492       14.552      

---
## 8 — Provider Comparison (Do sensors agree with each other?)

In [16]:
print('PM2.5 statistics by provider:')
prov = df[df['pm25'].notna()].groupby('provider_name')['pm25'].agg(
    avg='mean', median='median', std='std', min='min', max='max', count='count'
).round(2)
print(prov.to_string())

print('\nAQI category split by provider (%):')
for p in df['provider_name'].unique():
    sub = df[(df['provider_name']==p) & (df['aqi_cat']!='Unknown')]
    dist = sub['aqi_cat'].value_counts(normalize=True).mul(100).round(1).sort_index()
    print(f'\n{p}:')
    print(dist.to_string())

print('\nDo providers overlap in time? (shared hourly timestamps):')
for p in df['provider_name'].unique():
    sub = df[df['provider_name']==p]
    print(f'{p}: {sub["datetime"].min().date()} → {sub["datetime"].max().date()} ({len(sub):,} readings)')


PM2.5 statistics by provider:
                 avg  median     std  min     max   count
provider_name                                            
AirGradient    62.61   53.66   51.40  0.0  479.64   48897
AirNow         29.44   15.50   36.06  0.0  396.00   49076
Clarity        41.89   14.53  115.42  0.0  999.40  432980

AQI category split by provider (%):

AirNow:
aqi_cat
1-Good                       38.8
2-Moderate                   40.9
3-Unhealthy for Sensitive     5.8
4-Unhealthy                  12.7
5-Hazardous                   1.7

Clarity:
aqi_cat
1-Good                       37.4
2-Moderate                   42.5
3-Unhealthy for Sensitive     9.4
4-Unhealthy                   6.5
5-Hazardous                   4.1

AirGradient:
aqi_cat
1-Good                       12.8
2-Moderate                   20.4
3-Unhealthy for Sensitive    19.1
4-Unhealthy                  40.8
5-Hazardous                   6.9

Do providers overlap in time? (shared hourly timestamps):
AirNow: 2020-04-0

---
# More Interesting Analyses

| # | Question |
|---|---|
| 9 | How many truly clean days does Almaty get per year? |
| 10 | What is the longest stretch of bad air without a break? |
| 11 | Is each winter getting worse than the last? (month × year table) |
| 12 | How many breathable hours does an average day have? |
| 13 | When do sudden dangerous spikes happen? |
| 14 | Do fine particles (PM1) track closely with PM2.5? |
| 15 | Which weeks of the year are the most dangerous? |


## 9 — How Many Truly Clean Days Per Year?
A day is 'clean' if the **daily average PM2.5 < 12 µg/m³** (WHO Good category).
We use the AirNow sensor — it has the longest continuous daily record.

In [17]:
# Build daily averages from AirNow sensor
airnow = df[df['provider_name'] == 'AirNow'].copy()
airnow['date'] = airnow['datetime'].dt.date
daily = airnow.groupby(['year', 'date'])['pm25'].mean().reset_index()
daily.columns = ['year', 'date', 'daily_pm25']

# Count clean / moderate / bad days per year
result = daily.groupby('year').apply(lambda x: pd.Series({
    'total_days'     : len(x),
    'clean_days'     : (x['daily_pm25'] < 12).sum(),
    'moderate_days'  : ((x['daily_pm25'] >= 12) & (x['daily_pm25'] < 35)).sum(),
    'bad_days'       : (x['daily_pm25'] >= 35).sum(),
    'pct_clean'      : round((x['daily_pm25'] < 12).mean() * 100, 1),
})).astype({'total_days':int, 'clean_days':int, 'moderate_days':int, 'bad_days':int})

print('Clean air days per year (AirNow sensor):')
print(result.to_string())

best_year  = result['pct_clean'].idxmax()
worst_year = result['pct_clean'].idxmin()
print(f'\nBest year:  {best_year} — {result.loc[best_year, "clean_days"]} clean days ({result.loc[best_year, "pct_clean"]}%)')
print(f'Worst year: {worst_year} — {result.loc[worst_year, "clean_days"]} clean days ({result.loc[worst_year, "pct_clean"]}%)')


Clean air days per year (AirNow sensor):
      total_days  clean_days  moderate_days  bad_days  pct_clean
year                                                            
2020         267         118             94        55       44.2
2021         365         211             85        69       57.8
2022         365          15            257        93        4.1
2023         365         140            135        90       38.4
2024         366         141            139        86       38.5
2025         318          81            179        58       25.5

Best year:  2021 — 211 clean days (57.8%)
Worst year: 2022 — 15 clean days (4.1%)


## 10 — Longest Stretch of Bad Air Without a Break
How many days in a row was the air unhealthy (daily avg > 35 µg/m³)?

In [18]:
daily_sorted = daily.sort_values('date').copy()
daily_sorted['is_bad'] = daily_sorted['daily_pm25'] > 35

# Count consecutive bad days
daily_sorted['streak_id'] = (daily_sorted['is_bad'] != daily_sorted['is_bad'].shift()).cumsum()
streaks = daily_sorted[daily_sorted['is_bad']].groupby('streak_id').agg(
    start    = ('date', 'first'),
    end      = ('date', 'last'),
    days     = ('date', 'count'),
    avg_pm25 = ('daily_pm25', 'mean'),
).sort_values('days', ascending=False)

print('Top 10 longest bad air streaks (daily avg PM2.5 > 35 µg/m³):')
print(streaks.head(10).round(1).to_string(index=False))

print('\nTop 10 longest clean streaks (daily avg PM2.5 < 12 µg/m³):')
daily_sorted['is_clean'] = daily_sorted['daily_pm25'] < 12
daily_sorted['clean_streak_id'] = (daily_sorted['is_clean'] != daily_sorted['is_clean'].shift()).cumsum()
clean_streaks = daily_sorted[daily_sorted['is_clean']].groupby('clean_streak_id').agg(
    start    = ('date', 'first'),
    end      = ('date', 'last'),
    days     = ('date', 'count'),
    avg_pm25 = ('daily_pm25', 'mean'),
).sort_values('days', ascending=False)
print(clean_streaks.head(10).round(1).to_string(index=False))


Top 10 longest bad air streaks (daily avg PM2.5 > 35 µg/m³):
     start        end  days  avg_pm25
2022-01-21 2022-03-03    42      92.1
2023-02-01 2023-03-12    40      90.0
2025-02-01 2025-02-25    25      79.9
2020-12-21 2021-01-12    23     139.6
2020-11-30 2020-12-19    20      99.0
2023-01-12 2023-01-30    19     108.4
2022-12-02 2022-12-16    15      81.8
2022-12-18 2022-12-29    12      76.7
2021-12-09 2021-12-19    11      60.9
2021-11-27 2021-12-07    11      64.7

Top 10 longest clean streaks (daily avg PM2.5 < 12 µg/m³):
     start        end  days  avg_pm25
2021-04-14 2021-06-06    54       9.6
2021-03-02 2021-04-08    38       7.0
2021-01-23 2021-02-28    37       9.0
2020-07-03 2020-08-05    34       7.2
2023-07-22 2023-08-17    27       8.7
2024-07-16 2024-08-09    25       9.3
2021-07-25 2021-08-12    19      10.3
2021-06-21 2021-07-08    18       8.5
2023-04-24 2023-05-07    14       7.6
2023-05-26 2023-06-08    14      10.2


## 11 — Is Each Winter Getting Worse? (Month × Year Table)
A table where rows = months, columns = years. Shows average PM2.5 for each month of each year.
Read down a column to see one year. Read across a row to see if January 2022 was worse than January 2021.

In [19]:
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

pivot = df[df['pm25'].notna()].groupby(['year','month'])['pm25'].mean().round(1).unstack('year')
pivot.index = pivot.index.map(month_names)

print('Average PM2.5 (µg/m³) per month per year:')
print('(Higher = worse air quality)')
print(pivot.to_string())

print('\nWorst month-year combination:')
worst = pivot.stack().idxmax()
print(f'  {worst[0]} {worst[1]}  —  {pivot.stack().max():.1f} µg/m³')

print('\nBest month-year combination:')
best = pivot.stack().idxmin()
print(f'  {best[0]} {best[1]}  —  {pivot.stack().min():.1f} µg/m³')


Average PM2.5 (µg/m³) per month per year:
(Higher = worse air quality)
year    2020  2021  2022  2023  2024   2025  2026
month                                            
Jan      NaN  83.1  46.7  79.7  60.1   54.1  56.8
Feb      NaN   9.0  98.6  98.9  49.9   59.7   NaN
Mar      NaN   8.5  29.5  43.6  26.5   30.0   NaN
Apr     12.8  11.3  18.5  15.1  46.2   23.7   NaN
May      9.0  10.0  33.0  10.8  43.2   18.6   NaN
Jun     11.6   8.0  33.0  11.9  31.7   27.2   NaN
Jul      7.3  12.0  33.0  10.7  34.7   48.6   NaN
Aug     12.3  12.6  33.0   8.6  32.1   53.7   NaN
Sep     13.5  13.6  28.6  11.0  22.0   60.2   NaN
Oct     26.9  17.4  22.0  16.5  26.6   88.4   NaN
Nov     47.8  50.9  40.2  29.2  37.1  171.2   NaN
Dec    116.1  56.5  74.6  46.0  51.4   80.3   NaN

Worst month-year combination:
  Nov 2025  —  171.2 µg/m³

Best month-year combination:
  Jul 2020  —  7.3 µg/m³


## 12 — How Many Breathable Hours Does an Average Day Have?
'Breathable' = PM2.5 < 35 µg/m³. Broken down by season and year.

In [20]:
df['is_breathable'] = df['pm25'] < 35

print('Average breathable hours per day by season:')
season_order = ['Winter','Spring','Summer','Autumn']
by_season = df[df['pm25'].notna()].groupby('season')['is_breathable'].mean().mul(24).round(1)
print(by_season.reindex(season_order).to_string())

print('\nAverage breathable hours per day by year:')
by_year = df[df['pm25'].notna()].groupby('year')['is_breathable'].mean().mul(24).round(1)
print(by_year.to_string())

print('\nAverage breathable hours per day by month:')
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
by_month = df[df['pm25'].notna()].groupby('month')['is_breathable'].mean().mul(24).round(1)
by_month.index = by_month.index.map(month_names)
print(by_month.to_string())


Average breathable hours per day by season:
season
Winter     9.7
Spring    20.5
Summer    21.6
Autumn    19.6

Average breathable hours per day by year:
year
2020    19.2
2021    20.2
2022    18.0
2023    17.8
2024    19.9
2025    17.2
2026     6.8

Average breathable hours per day by month:
month
Jan    10.5
Feb     9.1
Mar    17.8
Apr    21.1
May    21.5
Jun    21.8
Jul    21.5
Aug    21.5
Sep    21.7
Oct    20.0
Nov    15.9
Dec     9.4


## 13 — When Do Sudden Dangerous Spikes Happen?
A spike = PM2.5 jumps by more than 50 µg/m³ in a single hour.
Shows the time of day and month when air quality can suddenly turn dangerous.

In [21]:
# Use AirNow for clean, continuous readings
airnow2 = df[df['provider_name'] == 'AirNow'].sort_values('datetime').copy()
airnow2['prev_pm25'] = airnow2['pm25'].shift(1)
airnow2['jump'] = airnow2['pm25'] - airnow2['prev_pm25']

spikes = airnow2[airnow2['jump'] > 50].copy()

print(f'Total sudden spikes (jump > 50 µg/m³ in 1 hour): {len(spikes)}')

print('\nTop 15 biggest spikes:')
print(spikes.nlargest(15, 'jump')[['datetime','pm25','prev_pm25','jump']].round(1).to_string(index=False))

print('\nSpikes by hour of day (when are they most likely):')
print(spikes['hour'].value_counts().sort_index().to_string())

print('\nSpikes by month:')
print(spikes['month'].map(month_names).value_counts().to_string())


Total sudden spikes (jump > 50 µg/m³ in 1 hour): 211

Top 15 biggest spikes:
                 datetime  pm25  prev_pm25  jump
2023-07-09 07:00:00+00:00 248.0       12.0 236.0
2023-11-22 14:00:00+00:00 214.0       22.0 192.0
2021-12-16 16:00:00+00:00 255.0       74.0 181.0
2020-12-10 06:00:00+00:00 268.0       99.0 169.0
2022-12-21 07:00:00+00:00 214.0       51.0 163.0
2022-12-31 15:00:00+00:00 195.0       42.0 153.0
2023-01-16 05:00:00+00:00 342.0      191.0 151.0
2020-12-24 13:00:00+00:00 330.0      180.0 150.0
2024-12-27 06:00:00+00:00 177.0       27.0 150.0
2020-11-26 18:00:00+00:00 225.0       91.0 134.0
2022-12-13 08:00:00+00:00 197.0       64.5 132.5
2020-12-24 05:00:00+00:00 310.0      181.0 129.0
2022-12-28 06:00:00+00:00 179.0       50.0 129.0
2023-01-16 01:00:00+00:00 163.0       35.4 127.6
2023-01-25 04:00:00+00:00 143.0       17.0 126.0

Spikes by hour of day (when are they most likely):
hour
0      3
1      3
2      6
3      9
4     14
5     25
6     37
7     15
8     17
9

## 14 — Do Fine Particles (PM1) Track PM2.5 Closely?
PM1 = ultra-fine particles < 1µm. Only the AirGradient sensors measure this.
If PM1/PM2.5 ratio is high, most of the pollution is ultra-fine — more dangerous for lungs.

In [22]:
ag = df[(df['provider_name'] == 'AirGradient') & df['pm1'].notna() & df['pm25'].notna()].copy()

print(f'AirGradient readings with both PM1 and PM2.5: {len(ag)}')

# Ratio: what fraction of PM2.5 is ultra-fine PM1
ag['pm1_ratio'] = ag['pm1'] / ag['pm25']
ag = ag[ag['pm1_ratio'] <= 1.5]   # remove sensor glitches

print(f'\nCorrelation between PM1 and PM2.5: {ag["pm1"].corr(ag["pm25"]):.3f}')
print(f'On average, PM1 makes up {ag["pm1_ratio"].mean()*100:.1f}% of PM2.5')

print('\nPM1/PM2.5 ratio by season (higher = more ultra-fine pollution):')
print(ag.groupby('season')['pm1_ratio'].mean().mul(100).round(1).sort_values(ascending=False).to_string())

print('\nPM1/PM2.5 ratio by hour:')
print(ag.groupby('hour')['pm1_ratio'].mean().mul(100).round(1).to_string())

print('\nAverage PM1 and PM2.5 side by side by month:')
month_comparison = ag.groupby('month')[['pm1','pm25']].mean().round(1)
month_comparison.index = month_comparison.index.map(month_names)
print(month_comparison.to_string())


AirGradient readings with both PM1 and PM2.5: 48897

Correlation between PM1 and PM2.5: 0.995
On average, PM1 makes up 54.3% of PM2.5

PM1/PM2.5 ratio by season (higher = more ultra-fine pollution):
season
Spring    63.1
Summer    60.9
Autumn    59.9
Winter    53.0

PM1/PM2.5 ratio by hour:
hour
0     49.2
1     49.3
2     50.6
3     52.5
4     53.2
5     54.2
6     54.9
7     55.4
8     55.6
9     56.3
10    56.6
11    56.9
12    56.9
13    56.8
14    56.8
15    56.5
16    56.3
17    55.9
18    55.1
19    53.4
20    54.3
21    53.3
22    51.9
23    49.8

Average PM1 and PM2.5 side by side by month:
        pm1  pm25
month            
Jan    31.3  56.6
Feb    35.2  63.3
Mar    27.7  47.3
Apr    16.6  26.2
May    10.6  16.3
Jun    10.4  16.3
Jul    10.0  14.9
Aug     6.9  10.6
Sep    12.1  19.5
Oct    26.5  45.0
Nov    29.1  49.8
Dec    44.9  79.9


## 15 — Which Weeks of the Year Are Most Dangerous?
Ranks all 52 weeks by their average PM2.5 across all years.

In [23]:
df['week'] = df['datetime'].dt.isocalendar().week.astype(int)

weekly = df[df['pm25'].notna()].groupby('week')['pm25'].agg(
    avg_pm25        = 'mean',
    pct_unhealthy   = lambda x: round((x > 55).mean() * 100, 1),
).round(2).sort_values('avg_pm25', ascending=False)

print('All 52 weeks ranked by average PM2.5 (worst first):')
print(weekly.to_string())

print(f'\nDirtiest week of the year: Week {weekly.index[0]}  ({weekly["avg_pm25"].iloc[0]:.1f} µg/m³ avg)')
print(f'Cleanest week of the year: Week {weekly.index[-1]}  ({weekly["avg_pm25"].iloc[-1]:.1f} µg/m³ avg)')
print(f'\nWeeks 1-8 (Jan-Feb) average:  {weekly.loc[weekly.index.isin(range(1,9)), "avg_pm25"].mean():.1f} µg/m³')
print(f'Weeks 22-35 (Jun-Aug) average: {weekly.loc[weekly.index.isin(range(22,36)), "avg_pm25"].mean():.1f} µg/m³')


All 52 weeks ranked by average PM2.5 (worst first):
      avg_pm25  pct_unhealthy
week                         
53      141.46           99.4
52       80.46           65.1
6        79.96           59.8
3        76.69           42.5
8        68.02           52.4
1        61.43           40.4
48       60.63           26.5
2        59.27           48.5
41       53.81            8.6
46       53.30           16.4
50       52.64           30.8
47       51.74           17.7
7        51.63           37.1
5        49.59           25.9
51       48.56           25.8
44       48.19           14.1
32       45.71            6.5
45       45.23           12.9
43       44.65           10.4
42       44.19            8.8
4        44.16           22.5
31       43.17            5.0
9        41.40           26.0
33       40.66            6.0
27       40.60            5.1
38       40.20            4.7
34       39.94            5.7
36       39.66            5.0
29       39.62            5.2
30       39.49    

---
# Задание: три человеческих вопроса + поймай враньё

| # | Вопрос |
|---|---|
| Q1 | Есть ли в Алматы настоящие чистые дни зимой — или это миф? |
| Q2 | Воздух ухудшается каждый год, или 2022 был случайным провалом? |
| Q3 | Холод вызывает загрязнение — или они просто оба хуже зимой? |
| ⚠️ | Поймай враньё: где данные почти солгали |


## Q1 — Есть ли в Алматы настоящие чистые дни зимой?
Зима в Алматы — котловина, уголь, смог. Бывает ли PM2.5 < 12 µg/m³ хоть иногда?

In [ ]:
import pandas as pd, os, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(os.path.join(path, 'air_quality_data.csv'))
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df['year']  = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['hour']  = df['datetime'].dt.hour

# Дневные средние по AirNow (самый длинный непрерывный ряд)
airnow = df[df['provider_name'] == 'AirNow'].copy()
airnow['date'] = airnow['datetime'].dt.date
daily = airnow.groupby(['year', 'month', 'date'])['pm25'].mean().reset_index()
daily.columns = ['year', 'month', 'date', 'daily_pm25']

winter = daily[daily['month'].isin([12, 1, 2])]
clean_winter = winter[winter['daily_pm25'] < 12]

print(f'Зимних дней в датасете: {len(winter)}')
print(f'Из них PM2.5 < 12 µg/m³: {len(clean_winter)} ({len(clean_winter)/len(winter)*100:.1f}%)')
print(f'\nСписок «чистых» зимних дней:')
print(clean_winter.sort_values('daily_pm25').to_string(index=False))

# Проверка подозрительных значений
print('\n--- Проверка: смотрим на часовые данные в «самый чистый» день ---')
cleanest_day = clean_winter.sort_values('daily_pm25').iloc[1]['date']  # второй по чистоте
check = airnow[airnow['datetime'].dt.date == cleanest_day][['datetime','pm25']]
print(f'День: {cleanest_day}')
print(check.to_string(index=False))
print('\n→ Одинаковые значения на протяжении многих часов подряд = завис датчик, не чистый воздух')


## Q2 — Воздух ухудшается каждый год, или 2022 был случайным провалом?
2022 выглядит как аномально плохой год. Может, без него тренд исчезает?

In [ ]:
annual = df[df['pm25'].notna()].groupby('year')['pm25'].mean().round(2)
print('Средний PM2.5 по годам:')
print(annual.to_string())

# Корреляция год vs PM2.5 (рост = ухудшение)
years = pd.Series(annual.index, index=annual.index)
r_with    = annual.corr(years)
r_without = annual.drop(2022).corr(years.drop(2022))

print(f'\nСила тренда (год vs PM2.5):')
print(f'  С 2022:    r = {r_with:.3f}')
print(f'  Без 2022:  r = {r_without:.3f}')
print('\n→ Без 2022 тренд стал СИЛЬНЕЕ — значит 2022 не «тянул» вверх,')
print('  а был ниже линии тренда. Ухудшение реально и не зависит от одного года.')

print('\n--- Проверка: сколько строк за каждый год? ---')
rows_per_year = df[df['pm25'].notna()].groupby('year')['pm25'].count()
print(rows_per_year.to_string())
print('\n→ 2024-2025 имеют в 25+ раз больше строк — Clarity сеть добавилась в 2024')
print('  Это не меняет вывод, но объясняет почему цифры более устойчивы с 2024')


## Q3 — Холод вызывает загрязнение, или они оба хуже зимой?
Общая корреляция температуры и PM2.5: r = −0.267. Но может, это просто «зима»?

In [ ]:
# Данные AirGradient — единственные с температурой
ag = df[(df['provider_name'] == 'AirGradient') & df['temperature'].notna() & df['pm25'].notna()]

r_global = ag['temperature'].corr(ag['pm25'])
print(f'Глобальная корреляция температура vs PM2.5: r = {r_global:.3f}')
print('(Выглядит убедительно: холоднее → грязнее)\n')

# Теперь внутри каждого месяца
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('Корреляция ВНУТРИ каждого месяца (контролируем сезонность):')
for m in sorted(ag['month'].unique()):
    sub = ag[ag['month'] == m]
    r = sub['temperature'].corr(sub['pm25'])
    note = '← почти нет связи' if abs(r) < 0.15 else ''
    print(f'  {month_names[m]:>3}: r = {r:>6.3f}  (n={len(sub):,}) {note}')

print('\n→ Внутри большинства месяцев корреляция близка к нулю')
print('→ Глобальный r = −0.267 — это не «холод вызывает смог»,')
print('  а «зима приносит и холод, и отопление, и загрязнение» — одновременно')
print('  Настоящий виновник — отопительный сезон, а не температура сама по себе')


---
## ⚠️ Поймай враньё: AirGradient «врёт» или это мы сравниваем несравнимое?

**Убедительный вывод:** *«AirGradient показывает 62.6 µg/m³, AirNow — 29.4 µg/m³. Датчики AirGradient явно завышают — им нельзя доверять»*

In [ ]:
print('=== КРАСИВЫЙ ВЫВОД ===')
for p in df['provider_name'].unique():
    avg = df[df['provider_name']==p]['pm25'].mean()
    print(f'  {p}: {avg:.2f} µg/m³')
print('Звучит как: AirGradient в 2x хуже — сломан?\n')

print('=== ПРОВЕРКА №1: когда каждый провайдер измерял? ===')
for p in df['provider_name'].unique():
    sub = df[df['provider_name']==p]
    print(f'  {p}: {sub["datetime"].min().date()} → {sub["datetime"].max().date()}')

print('\n=== ПРОВЕРКА №2: все три провайдера в одном и том же периоде (ноябрь-январь) ===')
nov_jan = df[df['month'].isin([11, 12, 1])]
same_period = nov_jan.groupby('provider_name')['pm25'].agg(avg='mean', count='count').round(2)
print(same_period.to_string())

print('\n=== ПРОВЕРКА №3: AirNow в тот же период что AirGradient (с ноября 2024) ===')
airnow_late = df[(df['provider_name']=='AirNow') & (df['datetime'].dt.date >= pd.Timestamp('2024-11-01').date())]
ag_data = df[df['provider_name']=='AirGradient']
print(f'  AirNow (ноябрь 2024+):  {airnow_late["pm25"].mean():.2f} µg/m³  (строк: {len(airnow_late)})')
print(f'  AirGradient (весь ряд): {ag_data["pm25"].mean():.2f} µg/m³  (строк: {len(ag_data)})')

print('\n=== ПРАВДА РЯДОМ ===')
print('AirGradient измерял ТОЛЬКО с ноября 2024 — самые грязные месяцы в году')
print('В тот же ноябрь-январь AirNow тоже показывает ~57 µg/m³, а не 29')
print('Разрыв 62.6 vs 29.4 — не «сломанный датчик», а сравнение зимы с целым годом')
print('\n→ Прежде чем обвинять датчик — проверь, за какой период он считал')


---
## Одно открытие — одной фразой

> **Из 42 «чистых» зимних дней минимум 2 — это завис датчик:**
> **PM2.5 = 0.102 ровно 30 часов подряд — в реальности такого не бывает.**

**Проверка:** часовые данные за 13–14 января 2022 показывают одно и то же значение без единого отклонения. Вывод: реальных чистых зимних дней ещё меньше, чем кажется.

---
# Питч (30 секунд)

Мы взяли данные о качестве воздуха в Алматы — 544 тысячи измерений с 2020 по 2026.

**Открытие:** зимой в Алматы почти нет чистого воздуха. Из 451 зимнего дня только 42 были ниже нормы ВОЗ. И даже среди них мы нашли ловушку: два «самых чистых» дня — это зависший датчик, который 30 часов подряд выдавал одно и то же число.

**Где данные почти солгали:** AirGradient показывает 62.6 µg/m³, AirNow — 29.4. Напрашивается вывод: «датчик врёт». Но стоило проверить период — AirGradient измерял только ноябрь–январь, а в эти же месяцы AirNow показывает 57. Разрыв был не в датчиках — а в том, что мы сравнивали зиму с целым годом.

**Главное:** самые убедительные числа — те, что надо проверять первыми.